CREATE DATABASE shipping_db;
USE shipping_db;
CREATE TABLE shipments (
shipment_id INT PRIMARY KEY,
port VARCHAR(50),
shipment_date DATE,
status VARCHAR(20)
);
CREATE TABLE shipment_delay (
shipment_id INT,
delay_reason VARCHAR(100),
delay_days INT
);
INSERT INTO shipments VALUES
(1,'Mumbai','2024-01-01','Delivered'),
(2,'Chennai','2024-01-02','Delayed'),
(3,'Mumbai','2024-01-03','Delivered'),
(4,'Kolkata','2024-01-04','Delayed'),
(5,'Chennai','2024-01-05','Delivered');
INSERT INTO shipment_delay VALUES
(2,'Weather',2),
(4,'Port Congestion',3);

Python Libraries

In [ ]:
!pip install pandas mysql-connector-python matplotlib openai

Connect Python to MySQL

In [ ]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
host="localhost",
user="root",
password="",
database="shipping_db"
)

query = """
SELECT 
s.shipment_id,
s.port,
s.status,
d.delay_reason
FROM shipments s
LEFT JOIN shipment_delay d
ON s.shipment_id = d.shipment_id
"""

df = pd.read_sql(query, conn)

print(df)

Calculate Delay Rate per Port

Total Shipments

In [ ]:
total_shipments = df.groupby("port")["shipment_id"].count()

Delayed Shipments

In [ ]:
delayed = df[df["status"]=="Delayed"]
delayed_shipments = delayed.groupby("port")["shipment_id"].count()

Delay Rate

In [ ]:
delay_rate = (delayed_shipments / total_shipments) * 100
delay_rate = delay_rate.fillna(0)
print(delay_rate)

Analyze Delay Reasons

In [ ]:
reason_counts = delayed["delay_reason"].value_counts()
print(reason_counts)

Visualization

In [ ]:
import matplotlib.pyplot as plt
delay_rate.plot(kind="bar")
plt.title("Shipping Delay Rate by Port")
plt.xlabel("Port")
plt.ylabel("Delay Rate (%)")
plt.show()

Delay Reasons Chart

In [ ]:
reason_counts.plot(kind="bar")
plt.title("Shipping Delay Reasons")
plt.xlabel("Reason")
plt.ylabel("Number of Delays")
plt.show()

In [ ]:
from openai import OpenAI

client = OpenAI()

summary = delay_rate.to_string()

response = client.responses.create(
model="gpt-4.1-mini",
input=f"""
Analyze shipping delay rates by port and suggest operational improvements:

{summary}
"""
)

print(response.output_text)